# 06 — Interactive Demo
### DiscoverAI · Deloitte x LUISS

Gradio web interface integrating the full DiscoverAI pipeline.

Prerequisites: run notebooks 01-04 first.
Notebook 05 optional (summaries shown if product_profiles.csv exists).

Tabs:
- Search: free text query with price/rating filters and BART summaries
- Similar products: content-based recommendation
- System info: index stats, model info, coverage


## 0 . Setup

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import faiss
import gradio as gr
import torch
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')

DATA_DIR = '/Users/danielegiovanardi/MSTERRORS/Mean-Squared-Terrors/data'

comb_emb = np.load(os.path.join(DATA_DIR, 'combined_embeddings.npy'))
index    = faiss.read_index(os.path.join(DATA_DIR, 'faiss_index.bin'))

profiles_path = os.path.join(DATA_DIR, 'product_profiles.csv')
index_path    = os.path.join(DATA_DIR, 'embedding_index_enriched.csv')
idx = pd.read_csv(profiles_path if os.path.exists(profiles_path) else index_path)

if torch.backends.mps.is_available():   DEVICE = 'mps'
elif torch.cuda.is_available():          DEVICE = 'cuda'
else:                                    DEVICE = 'cpu'

model = SentenceTransformer('all-mpnet-base-v2', device=DEVICE)
has_summaries = 'summary_full' in idx.columns and idx['summary_full'].notna().any()
has_entities  = 'ingredients' in idx.columns and idx['ingredients'].notna().any()

print(f'Device: {DEVICE}  |  Prodotti: {len(idx):,}')
print(f'Summaries: {"OK" if has_summaries else "NO"}  |  Entities: {"OK" if has_entities else "NO"}')


## 1 . Quality scores

Computed here if product_profiles.csv does not already contain them.

In [ ]:
if 'quality_score' not in idx.columns:
    reviews = pd.read_csv(os.path.join(DATA_DIR, 'reviews_cleaned.csv'))
    rev_agg = reviews.groupby('parent_asin').agg(
        n_reviews=('rating','count'),
        pct_positive=('rating', lambda x: (x>=4).mean()),
        pct_negative=('rating', lambda x: (x<=2).mean()),
        total_helpful=('helpful_vote','sum'),
    ).reset_index()
    idx = idx.merge(rev_agg, on='parent_asin', how='left')
    rating_norm  = (idx['product_avg_rating'].fillna(3)-1)/4
    sentiment    = idx['pct_positive'].fillna(0.5) - 0.5*idx['pct_negative'].fillna(0.5)
    hv_log       = np.log1p(idx['total_helpful'].fillna(0))
    helpful_cred = (hv_log/hv_log.quantile(0.95)).clip(upper=1.0)
    idx['quality_score']    = 0.4*rating_norm + 0.35*sentiment + 0.25*helpful_cred
    rc_log = np.log1p(idx['product_rating_count'].fillna(0))
    idx['popularity_score'] = (rc_log/rc_log.quantile(0.95)).clip(upper=1.0)
    for col in ['quality_score','popularity_score']:
        mn, mx = idx[col].min(), idx[col].max()
        idx[col] = (idx[col]-mn)/(mx-mn+1e-9)
print('Quality scores pronti.')


## 2 . Search functions (copied from notebook 04)

All four cells below are identical to notebook 04.

In [ ]:
# Mappa parole → bucket di prezzo
PRICE_INTENT = {
    "cheap":        ["budget", "low"],
    "affordable":   ["budget", "low"],
    "budget":       ["budget"],
    "inexpensive":  ["budget", "low"],
    "low cost":     ["budget", "low"],
    "low-cost":     ["budget", "low"],
    "value":        ["budget", "low", "mid"],
    "premium":      ["high", "premium"],
    "luxury":       ["premium"],
    "expensive":    ["high", "premium"],
    "professional": ["high", "premium"],
    "high end":     ["high", "premium"],
    "high-end":     ["high", "premium"],
    "best":         None,   # "best" non implica prezzo — solo qualità
}

# Mappa parole → boost qualità (non prezzo)
QUALITY_BOOST_WORDS = {"best", "top", "highest rated", "most popular"}

def parse_query(query: str):
    """
    Analizza la query e restituisce:
    - clean_query: query senza parole di prezzo (da encodare)
    - price_buckets: lista di bucket accettabili (None = nessun filtro)
    - quality_boost: se True, penalizza prodotti con rating basso nel re-ranking
    """
    q_lower = query.lower().strip()
    price_buckets = None
    quality_boost = False
    clean = q_lower

    for word, buckets in PRICE_INTENT.items():
        if word in q_lower:
            if buckets is not None:
                price_buckets = buckets
            else:
                quality_boost = True
            clean = clean.replace(word, "").strip()

    for word in QUALITY_BOOST_WORDS:
        if word in q_lower:
            quality_boost = True
            clean = clean.replace(word, "").strip()

    # Pulizia spazi doppi
    clean = re.sub(r"\s+", " ", clean).strip()
    # Se la pulizia ha svuotato la query, usa quella originale
    if len(clean) < 3:
        clean = query

    return {
        "original":      query,
        "clean":         clean,
        "price_buckets": price_buckets,
        "quality_boost": quality_boost,
    }

# Test del parser
test_queries = [
    "affordable moisturizer for sensitive skin",
    "cheap sunscreen SPF 50",
    "premium anti-aging serum vitamin C",
    "best electric toothbrush whitening",
    "baby lotion fragrance free",
    "budget protein powder chocolate",
]

print("Test query parser:")
print(f"{'Query originale':<45} {'Clean':<35} {'Price buckets':<20} {'Quality boost'}")
print("─" * 115)
for q in test_queries:
    parsed = parse_query(q)
    print(f"  {q:<43} {parsed['clean']:<35} "
          f"{str(parsed['price_buckets']):<20} {parsed['quality_boost']}")


In [ ]:
# Parametri re-ranking
BETA_QUALITY    = 0.12   # peso quality score nello score finale
BETA_POPULARITY = 0.05   # peso popularity (piccolo — non vogliamo solo i prodotti famosi)
N_CANDIDATES    = 50     # quanti candidati recuperare da FAISS prima del re-ranking

def search(
    query: str,
    k: int = 5,
    price_buckets: list = None,   # override manuale (sovrascrive il parser)
    min_rating: float = None,     # filtro hard su rating minimo
    beta_quality: float = BETA_QUALITY,
    beta_popularity: float = BETA_POPULARITY,
    verbose: bool = False,
):
    """
    Cerca i prodotti più rilevanti per una query in linguaggio naturale.

    Args:
        query         : testo libero
        k             : numero di risultati
        price_buckets : filtro prezzo manuale (override del parser)
        min_rating    : filtro hard su product_avg_rating (es. 3.5)
        beta_quality  : peso del quality score nel re-ranking
        beta_popularity: peso della popularity nel re-ranking
        verbose       : mostra dettagli dello score
    """
    # 1. Parse
    parsed = parse_query(query)
    buckets = price_buckets if price_buckets is not None else parsed["price_buckets"]
    boost   = parsed["quality_boost"]

    if verbose:
        print(f"Query originale : {parsed['original']}")
        print(f"Query pulita    : {parsed['clean']}")
        print(f"Price buckets   : {buckets}")
        print(f"Quality boost   : {boost}")
        print()

    # 2. Encoding
    q_vec = model.encode(
        [parsed["clean"]],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)

    # 3. FAISS retrieval — più candidati se c'è filtro (alcuni verranno scartati)
    n_cand = N_CANDIDATES * 3 if buckets else N_CANDIDATES
    D, I   = index.search(q_vec, n_cand)

    # 4. Costruisci DataFrame risultati
    res = idx.iloc[I[0]].copy()
    res["similarity"] = D[0]

    # 5. Filtro hard price_bucket
    if buckets:
        res = res[res["price_bucket"].isin(buckets)]

    # 6. Filtro hard min_rating
    if min_rating is not None:
        res = res[res["product_avg_rating"] >= min_rating]

    # 7. Quality boost se la query contiene "best/top/highest rated"
    eff_beta_quality = beta_quality * 1.5 if boost else beta_quality

    # 8. Score finale
    res["score"] = (
        res["similarity"] +
        eff_beta_quality  * res["quality_score"].fillna(0) +
        beta_popularity   * res["popularity_score"].fillna(0)
    )

    # 9. Sort e top-K
    res = res.sort_values("score", ascending=False).head(k)

    cols = ["product_title", "brand", "price", "price_bucket",
            "product_avg_rating", "n_reviews",
            "similarity", "quality_score", "popularity_score", "score"]
    return res[cols].reset_index(drop=True)

print("Funzione search() pronta.")
print(f"Parametri default: β_quality={BETA_QUALITY}, β_popularity={BETA_POPULARITY}, N_candidates={N_CANDIDATES}")


In [ ]:
import re as _re

# Pattern di negazione
NEGATION_PATTERNS = [
    r"without\s+(\w+(?:\s+\w+)?)",
    r"\bno\s+(\w+)",
    r"free\s+from\s+(\w+(?:\s+\w+)?)",
    r"(\w+)[\-\s]free\b",
]

def parse_query_v2(query):
    """
    Parser esteso con negation filter.
    Aggiunge 'exclude_words' alla risposta del parser originale.
    """
    q_lower = query.lower().strip()
    price_buckets, quality_boost = None, False
    clean = q_lower
    exclude_words = []

    # Estrai parole da escludere PRIMA di fare altre sostituzioni
    for pattern in NEGATION_PATTERNS:
        matches = _re.findall(pattern, q_lower)
        for m in matches:
            word = m.strip() if isinstance(m, str) else (m[0] if m else "")
            if word and len(word) > 2:
                exclude_words.append(word.lower())

    # Price intent (stesso dizionario di prima)
    PRICE_INTENT = {
        "cheap": ["budget","low"], "affordable": ["budget","low"],
        "budget": ["budget"], "inexpensive": ["budget","low"],
        "low cost": ["budget","low"], "low-cost": ["budget","low"],
        "value": ["budget","low","mid"], "premium": ["high","premium"],
        "luxury": ["premium"], "expensive": ["high","premium"],
        "professional": ["high","premium"], "high end": ["high","premium"],
        "high-end": ["high","premium"],
    }
    QUALITY_BOOST_WORDS = {"best","top","highest rated","most popular"}

    for word, buckets in PRICE_INTENT.items():
        if word in q_lower:
            price_buckets = buckets
            clean = clean.replace(word, "").strip()
    for word in QUALITY_BOOST_WORDS:
        if word in q_lower:
            quality_boost = True
            clean = clean.replace(word, "").strip()

    clean = _re.sub(r"\s+", " ", clean).strip()
    if len(clean) < 3:
        clean = query

    return {
        "original":      query,
        "clean":         clean,
        "price_buckets": price_buckets,
        "quality_boost": quality_boost,
        "exclude_words": exclude_words,
    }

def search_v2(
    query, k=5, price_buckets=None, min_rating=None,
    beta_quality=0.12, beta_popularity=0.05,
    verbose=False
):
    """
    Search aggiornata con negation filter.
    Sostituisce la funzione search() originale.
    """
    parsed  = parse_query_v2(query)
    buckets = price_buckets if price_buckets is not None else parsed["price_buckets"]
    boost   = parsed["quality_boost"]
    exclude = parsed["exclude_words"]

    if verbose:
        print(f"Query pulita  : {parsed['clean']}")
        print(f"Price buckets : {buckets}")
        print(f"Quality boost : {boost}")
        print(f"Escludi       : {exclude}")

    q_vec  = model.encode([parsed["clean"]], normalize_embeddings=True,
                           convert_to_numpy=True).astype(np.float32)
    n_cand = 150 if (buckets or exclude) else 50
    D, I   = index.search(q_vec, n_cand)

    res = idx.iloc[I[0]].copy()
    res["similarity"] = D[0]

    # Filtri hard
    if buckets:
        res = res[res["price_bucket"].isin(buckets)]
    if min_rating:
        res = res[res["product_avg_rating"] >= min_rating]

    # Negation filter: escludi prodotti che contengono le parole da escludere
    if exclude:
        for word in exclude:
            # Controlla nel titolo
            title_mask = ~res["product_title"].str.lower().str.contains(
                word, na=False, regex=False
            )
            res = res[title_mask]

    eff_beta = beta_quality * 1.5 if boost else beta_quality
    res["score"] = (
        res["similarity"] +
        eff_beta  * res["quality_score"].fillna(0) +
        beta_popularity * res["popularity_score"].fillna(0)
    )
    return res.sort_values("score", ascending=False).head(k).reset_index(drop=True)

# ── Test negation filter ──────────────────────────────────────────────────────
print("Test negation filter:\n")
test_neg_queries = [
    "sleep supplement without melatonin",
    "moisturizer fragrance free sensitive skin",
    "shampoo sulfate-free dry hair",
    "sunscreen no oxybenzone",
]

for q in test_neg_queries:
    parsed = parse_query_v2(q)
    print(f"Query: '{q}'")
    print(f"  Escludi: {parsed['exclude_words']}")
    res = search_v2(q, k=5, verbose=False)
    for i, row in res.iterrows():
        title = str(row['product_title'])[:55]
        print(f"  {i+1}. [{row['score']:.4f}] ⭐{row['product_avg_rating']:.1f}  {title}")
    print()


In [ ]:
import re as _re

# ── Dizionario sinonimi per query implicite ───────────────────────────────────
SYNONYM_MAP = {
    # Sleep
    "help me sleep":        "sleep aid melatonin supplement",
    "can't sleep":          "sleep aid insomnia supplement",
    "fall asleep":          "sleep supplement melatonin",
    # Skin
    "dry cracked skin":     "moisturizer healing ointment dry skin",
    "fix dry skin":         "moisturizer cream dry skin",
    "protect skin sun":     "sunscreen SPF UV protection",
    "sun protection":       "sunscreen SPF broad spectrum",
    "at the beach":         "sunscreen waterproof SPF",
    # Hair
    "stop hair loss":       "hair loss treatment DHT blocker",
    "grow hair":            "hair growth supplement biotin",
    "longer stronger hair": "hair growth vitamins biotin collagen",
    # Energy
    "energy boost morning": "energy supplement caffeine B12",
    "feel tired":           "energy supplement fatigue",
    # Pain
    "joint pain":           "joint support glucosamine supplement pain relief",
    "knee pain":            "knee brace support joint pain relief",
    # Weight
    "lose weight":          "weight loss fat burner supplement",
    "burn fat":             "thermogenic fat burner supplement",
    # Teeth
    "clean teeth":          "teeth cleaning dental hygiene",
    "fresh breath":         "fresh breath mouthwash oral hygiene",
    # Gut
    "digestive problems":   "probiotic digestive supplement",
    "bloating":             "probiotic digestive enzyme supplement",
}

def expand_query(query):
    """Espande query implicite con termini del dominio."""
    q_lower = query.lower().strip()
    for pattern, expansion in SYNONYM_MAP.items():
        if pattern in q_lower:
            # Sostituisce il pattern con l'espansione arricchita
            expanded = q_lower.replace(pattern, expansion)
            return expanded
    return query

# ── Dosage filter ─────────────────────────────────────────────────────────────
DOSAGE_PATTERN = _re.compile(
    r'\b(\d+\s*(?:mg|mcg|iu|g|ml|oz|lb|%|spf\s*\d+|x\d+))\b',
    _re.IGNORECASE
)

def extract_dosages(text):
    """Estrae numeri con unità di misura da una query."""
    return [m.group(0).lower().replace(' ', '') for m in DOSAGE_PATTERN.finditer(text)]

def dosage_filter(results_df, dosages):
    """Filtra i risultati che contengono almeno uno dei dosaggi richiesti."""
    if not dosages:
        return results_df
    mask = results_df['product_title'].str.lower().apply(
        lambda title: any(d in title.replace(' ', '') for d in dosages)
    )
    filtered = results_df[mask]
    # Se rimangono almeno 2 risultati usa il filtro, altrimenti ignora
    return filtered if len(filtered) >= 2 else results_df

# ── search_v3 — integra tutto ─────────────────────────────────────────────────
def search_v3(
    query,
    k=5,
    price_buckets=None,
    min_rating=3.5,           # default 3.5 — rimuove prodotti scadenti
    beta_quality=None,        # None = adattivo
    beta_popularity=0.05,
    verbose=False,
):
    """
    Search v3 con tutti i miglioramenti:
    - Synonym expansion per query implicite
    - Dosage filter (1000mg, SPF 50, ecc.)
    - Min rating default 3.5
    - β_quality adattivo: alto quando similarity è bassa
    - Negation filter
    """
    # 1. Synonym expansion
    expanded = expand_query(query)
    if expanded != query.lower().strip() and verbose:
        print(f"  Expansion: '{query}' → '{expanded}'")

    # 2. Parse + negation
    parsed   = parse_query_v2(expanded)
    buckets  = price_buckets if price_buckets is not None else parsed['price_buckets']
    boost    = parsed['quality_boost']
    exclude  = parsed['exclude_words']

    # 3. Estrai dosaggi dalla query ORIGINALE
    dosages  = extract_dosages(query)
    if dosages and verbose:
        print(f"  Dosaggi: {dosages}")

    # 4. Encoding
    q_vec = model.encode([parsed['clean']], normalize_embeddings=True,
                          convert_to_numpy=True).astype(np.float32)

    # 5. FAISS retrieval
    n_cand = 150 if (buckets or exclude or dosages) else 80
    D, I   = index.search(q_vec, n_cand)

    res = idx.iloc[I[0]].copy()
    res['similarity'] = D[0]

    # 6. Filtri hard
    if buckets:
        res = res[res['price_bucket'].isin(buckets)]
    if min_rating is not None:
        res = res[res['product_avg_rating'] >= min_rating]
    if exclude:
        for word in exclude:
            res = res[~res['product_title'].str.lower().str.contains(word, na=False, regex=False)]

    # 7. Dosage filter
    if dosages:
        res = dosage_filter(res, dosages)

    # 8. β adattivo: se similarity media è bassa, quality pesa di più
    if beta_quality is None:
        avg_sim = res['similarity'].mean() if len(res) > 0 else 0.6
        beta_quality = 0.20 if avg_sim < 0.65 else 0.12

    eff_beta = beta_quality * 1.5 if boost else beta_quality

    # 9. Score finale
    if len(res) == 0:
        return pd.DataFrame()

    res['score'] = (
        res['similarity'] +
        eff_beta  * res['quality_score'].fillna(0) +
        beta_popularity * res['popularity_score'].fillna(0)
    )
    return res.sort_values('score', ascending=False).head(k).reset_index(drop=True)

print("search_v3() pronta.")
print("  - min_rating default: 3.5")
print("  - β adattivo: 0.20 se sim<0.65, 0.12 altrimenti")
print("  - dosage filter attivo")
print("  - synonym expansion attiva")


## 3 . Recommendation function

In [ ]:
def recommend_similar(query, k=5):
    if query in idx['parent_asin'].values:
        source = idx[idx['parent_asin'] == query].iloc[0]
    else:
        q_vec = model.encode([query], normalize_embeddings=True,
                              convert_to_numpy=True).astype(np.float32)
        _, I  = index.search(q_vec, 1)
        source = idx.iloc[I[0][0]]
    emb_row = int(source['emb_row'])
    q_vec   = comb_emb[emb_row:emb_row+1]
    D, I    = index.search(q_vec, k+1)
    res = idx.iloc[I[0]].copy()
    res['similarity'] = D[0]
    res = res[res['parent_asin'] != source['parent_asin']].head(k).reset_index(drop=True)
    return source, res

print('Recommendation pronta.')


## 4 . HTML result formatters

In [ ]:
TABLE_CSS = (
    '<style>'
    '.rtbl{width:100%;border-collapse:collapse;font-size:13px}'
    '.rtbl th{padding:8px 10px;text-align:left;border-bottom:2px solid #e2e8f0;'
    'color:#718096;font-weight:500;font-size:12px}'
    '.rtbl td{padding:8px 10px;border-bottom:1px solid #f7fafc;vertical-align:top}'
    '.rtbl tr:hover td{background:#f7fafc}'
    '.sc{background:#ebf8ff;color:#2b6cb0;padding:2px 7px;border-radius:8px;'
    'font-size:11px;font-weight:600}'
    '.sm{font-size:11px;color:#718096;margin-top:4px;line-height:1.5}'
    '.pr{color:#276749;font-size:11px}'
    '.cn{color:#9b2c2c;font-size:11px}'
    '</style>'
)

def format_results(df):
    if df is None or len(df) == 0:
        return "<p style='color:#718096;padding:1rem'>Nessun risultato.</p>"
    html = TABLE_CSS + "<table class='rtbl'><tr><th>#</th><th>Prodotto</th>"
    html += "<th>Rating</th><th>Prezzo</th><th>Score</th></tr>"
    for i, row in df.iterrows():
        title  = str(row.get('product_title',''))[:65]
        brand  = str(row.get('brand','')) if pd.notna(row.get('brand')) else ''
        rating = f"&#11088; {row['product_avg_rating']:.1f}" if pd.notna(row.get('product_avg_rating')) else 'N/D'
        price  = f"${row['price']:.2f}" if pd.notna(row.get('price')) else '&#8212;'
        score  = f"{row.get('score', row.get('similarity',0)):.3f}"
        nrev   = f" <small style='color:#a0aec0'>({int(row['n_reviews'])}rev)</small>" if pd.notna(row.get('n_reviews')) else ''
        summ   = ''
        if has_summaries and pd.notna(row.get('summary_full')) and str(row.get('summary_full','')).strip():
            s = str(row['summary_full'])[:150]
            dots = '...' if len(str(row['summary_full']))>150 else ''
            summ += f"<div class='sm'>{s}{dots}</div>"
            if pd.notna(row.get('pros')) and str(row.get('pros','')).strip():
                summ += f"<div class='pr'>+ {str(row['pros'])[:100]}</div>"
            if pd.notna(row.get('cons')) and str(row.get('cons','')).strip():
                summ += f"<div class='cn'>- {str(row['cons'])[:100]}</div>"
        brd = f"<br><small style='color:#a0aec0'>{brand}</small>" if brand else ''
        html += f"<tr><td><b>{i+1}</b></td><td><b>{title}</b>{brd}{nrev}{summ}</td>"
        html += f"<td>{rating}</td><td style='color:#276749'>{price}</td>"
        html += f"<td><span class='sc'>{score}</span></td></tr>"
    html += '</table>'
    return html

def format_detail(asin):
    row = idx[idx['parent_asin'] == asin]
    if len(row) == 0: return ''
    row = row.iloc[0]
    h = f"<div style='padding:8px 0'>"
    h += f"<h3 style='margin:0 0 4px;font-size:15px'>{str(row['product_title'])[:100]}</h3>"
    if pd.notna(row.get('brand')): h += f"<p style='color:#718096;font-size:13px;margin:0'>Brand: {row['brand']}</p>"
    if pd.notna(row.get('price')): h += f"<p style='font-size:13px;margin:4px 0'>&#128176; ${row['price']:.2f} ({row.get('price_bucket','')})</p>"
    nrev = int(row['n_reviews']) if pd.notna(row.get('n_reviews')) else '?'
    h += f"<p style='font-size:13px;margin:4px 0'>&#11088; {row.get('product_avg_rating','?')} &middot; {nrev} review</p>"
    if has_summaries and pd.notna(row.get('summary_full')):
        h += f"<hr style='border:0.5px solid #e2e8f0;margin:8px 0'>"
        h += f"<p style='font-size:13px'><b>Riassunto:</b> {row['summary_full']}</p>"
        if pd.notna(row.get('pros')) and row['pros']: h += f"<p style='font-size:12px;color:#276749'>+ {row['pros']}</p>"
        if pd.notna(row.get('cons')) and row['cons']: h += f"<p style='font-size:12px;color:#9b2c2c'>- {row['cons']}</p>"
    if has_entities and pd.notna(row.get('ingredients')) and row['ingredients']:
        ings = str(row['ingredients']).replace('|',' · ')[:200]
        h += f"<p style='font-size:12px'><b>Ingredienti:</b> {ings}</p>"
    h += '</div>'
    return h

print('Formatters pronti.')


## 5 . Gradio UI layout

Three tabs: Search, Similar products, System info.

In [ ]:
def do_search(query, n, price, rating_s, show_sc):
    if not query.strip(): return '<p>Inserisci una query.</p>', ''
    parsed   = parse_query_v2(query)
    expanded = expand_query(query)
    buckets  = [price.lower()] if price != 'Tutti' else parsed['price_buckets']
    min_r    = float(rating_s) if rating_s != 'Nessuno' else None
    results  = search_v3(query, k=int(n), price_buckets=buckets, min_rating=min_r)
    if len(results) == 0:
        return "<p style='color:#718096'>Nessun risultato. Prova ad abbassare il rating minimo.</p>", ''
    info = "<div style='font-size:12px;color:#718096;margin-bottom:8px;padding:6px 10px;background:#f7fafc;border-radius:6px'>"
    if expanded.lower().strip() != query.lower().strip():
        info += f'Espansione: <i>{query}</i> &#8594; <i>{expanded}</i>&nbsp;&nbsp;'
    if buckets: info += f'Prezzo: {buckets}&nbsp;&nbsp;'
    if parsed['quality_boost']: info += 'Quality boost&nbsp;&nbsp;'
    if parsed['exclude_words']: info += f"Escludi: {parsed['exclude_words']}"
    info += '</div>'
    detail = format_detail(results.iloc[0]['parent_asin'])
    return info + format_results(results), detail

def do_recommend(q):
    if not q.strip(): return '<p>Inserisci un prodotto.</p>'
    source, recs = recommend_similar(q, k=5)
    if len(recs) == 0: return '<p>Nessuna raccomandazione.</p>'
    recs['score'] = recs['similarity']
    return f"<p style='font-size:13px'><b>Simili a:</b> {str(source['product_title'])[:80]}</p>" + format_results(recs)

with gr.Blocks(title='DiscoverAI', theme=gr.themes.Soft(),
               css='.gradio-container{max-width:1100px!important;margin:auto}') as demo:

    gr.HTML('<div style="text-align:center;padding:16px 0 8px">'
            '<h1 style="font-size:26px;margin:0">DiscoverAI</h1>'
            '<p style="color:#718096;margin:4px 0 0;font-size:14px">'
            'Semantic Product Search &middot; Health &amp; Personal Care &middot; Deloitte &times; LUISS 2026</p>'
            '</div>')

    with gr.Tabs():
        with gr.Tab('Search'):
            with gr.Row():
                with gr.Column(scale=4):
                    qbox = gr.Textbox(label='Query', placeholder='es. affordable moisturizer for sensitive skin...', lines=1)
                with gr.Column(scale=1):
                    sbtn = gr.Button('Cerca', variant='primary')
            with gr.Row():
                nres = gr.Slider(3, 10, value=5, step=1, label='Risultati')
                pdd  = gr.Dropdown(['Tutti','Budget','Low','Mid','High','Premium'], value='Tutti', label='Prezzo')
                rdd  = gr.Dropdown(['Nessuno','3.0','3.5','4.0','4.5'], value='3.5', label='Rating min')
                scb  = gr.Checkbox(label='Mostra score', value=False)
            rout = gr.HTML()
            gr.Markdown('#### Dettaglio primo risultato')
            dout = gr.HTML()
            sbtn.click(do_search, [qbox,nres,pdd,rdd,scb], [rout,dout])
            qbox.submit(do_search, [qbox,nres,pdd,rdd,scb], [rout,dout])
            gr.Examples(examples=[
                ['affordable moisturizer for sensitive skin'],
                ['best electric toothbrush whitening'],
                ['natural sleep supplement without melatonin'],
                ['baby lotion fragrance free'],
                ['vitamin C 1000mg supplement'],
                ['pain relief cream for joints arthritis'],
                ['budget sunscreen SPF 50'],
                ['help me sleep naturally'],
                ['protect skin from sun at the beach'],
                ['grow longer stronger hair'],
            ], inputs=qbox)

        with gr.Tab('Prodotti simili'):
            gr.Markdown('Inserisci il nome di un prodotto per trovare i piu simili nel catalogo.')
            rbox  = gr.Textbox(label='Prodotto', placeholder='es. Oral-B electric toothbrush...')
            rbtn  = gr.Button('Trova simili', variant='primary')
            rout2 = gr.HTML()
            rbtn.click(do_recommend, rbox, rout2)
            rbox.submit(do_recommend, rbox, rout2)
            gr.Examples(examples=[
                ['vitamin C supplement rose hips'],
                ['electric toothbrush rechargeable'],
                ['air purifier HEPA bedroom'],
                ['baby lotion Johnson'],
                ['pain relief cream arthritis'],
            ], inputs=rbox)

        with gr.Tab('Info sistema'):
            n_summ = int(idx['summary_full'].notna().sum()) if has_summaries else 0
            gr.HTML(
                '<div style="padding:16px;font-size:13px">'
                '<h3 style="font-size:16px;font-weight:500">Sistema DiscoverAI</h3>'
                '<table style="border-collapse:collapse;width:100%;max-width:600px">'
                f'<tr><td style="padding:5px 16px 5px 0;color:#718096">Prodotti</td><td><b>{len(idx):,}</b></td></tr>'
                '<tr><td style="padding:5px 16px 5px 0;color:#718096">Modello</td><td>all-mpnet-base-v2 &middot; 768 dim</td></tr>'
                '<tr><td style="padding:5px 16px 5px 0;color:#718096">FAISS</td><td>IndexFlatIP &middot; ricerca esatta</td></tr>'
                '<tr><td style="padding:5px 16px 5px 0;color:#718096">Re-ranking</td><td>similarity + quality + popularity</td></tr>'
                '<tr><td style="padding:5px 16px 5px 0;color:#718096">Query features</td><td>price intent &middot; negation &middot; synonyms &middot; dosage</td></tr>'
                '<tr><td style="padding:5px 16px 5px 0;color:#718096">Rating minimo</td><td>3.5 &#11088; (default)</td></tr>'
                f'<tr><td style="padding:5px 16px 5px 0;color:#718096">Summaries BART</td><td>{"OK - " + str(n_summ) + " prodotti" if has_summaries else "Non disponibili"}</td></tr>'
                f'<tr><td style="padding:5px 16px 5px 0;color:#718096">Entity recognition</td><td>{"OK" if has_entities else "Non disponibili"}</td></tr>'
                '</table></div>'
            )

print('Layout Gradio costruito.')


## 6 . Launch

Set share=True for a public Gradio link (valid 72h) — useful for Deloitte demo.

In [ ]:
demo.launch(share=True, inbrowser=True, server_port=7860, show_error=True)
